**Explanation Validator** </br></br>
The Explanation Validator module develops the skill of validating that AI explanations actually improve user understanding rather than assuming they do. Through A/B testing scenarios, users experience the same AI decision with and without explanations, then answer comprehension questions to measure understanding. The module reveals a critical insight: explanations don't automatically lead to better understanding—poorly designed explanations can confuse, overwhelm, or mislead users. Users practice identifying what makes explanations effective (clarity, relevance, actionability) versus ineffective (jargon, information overload, irrelevant details). This builds the essential skill of empirically validating explanation quality rather than relying on intuition.

In [ ]:
#@title pip install
from IPython.display import clear_output

!pip install ipywidgets
clear_output()  # Hides the output after the cell runs

In [ ]:
#@title Imports
import random
import time
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import Button, VBox, HBox, HTML, Layout, RadioButtons

In [ ]:
#@title Validation Scenarios
validation_scenarios = [
    {
        "title": "Loan Approval Decision",
        "decision": "Your loan application has been APPROVED for $15,000 at 7.2% APR",
        "versions": {
            "no_explanation": {
                "text": "Your loan application has been APPROVED for $15,000 at 7.2% APR. You can accept this offer in your online banking portal."
            },
            "good_explanation": {
                "text": "Your loan application has been APPROVED for $15,000 at 7.2% APR.\n\nKey factors in your approval:\n• Strong credit score (720)\n• Stable employment (5 years)\n• Low debt-to-income ratio (28%)\n\nYour interest rate (7.2%) is based on current market rates and your credit profile. Customers with similar credit scores typically receive rates between 6.8% - 7.5%.\n\nYou can accept this offer in your online banking portal."
            },
            "poor_explanation": {
                "text": "Your loan application has been APPROVED for $15,000 at 7.2% APR.\n\nModel Analysis Output:\n• Primary decision tree ensemble score: 0.847\n• Neural network auxiliary score: 0.823\n• Rule-based override check: PASS\n• Feature vector components: [720, 60000, 0.28, 5.2, 0, 0.15, ...]\n• FICO-v8 percentile: 68th\n• Debt service coverage ratio: 3.57\n• Weighted risk score: 0.18 (threshold: 0.25)\n• Model version: LoanDecision-v4.2.1\n• Training AUC: 0.91 (validation: 0.89)\n\nYou can accept this offer in your online banking portal."
            }
        },
        "comprehension_questions": [
            {
                "question": "Why was the loan approved?",
                "options": [
                    "Good credit score, stable employment, and low debt",
                    "The AI model scored it highly",
                    "Market rates are favorable",
                    "Not enough information to determine"
                ],
                "correct_with_good": "Good credit score, stable employment, and low debt",
                "correct_with_none": "Not enough information to determine",
                "correct_with_poor": "Not enough information to determine"
            },
            {
                "question": "What could you do to potentially get a better interest rate in the future?",
                "options": [
                    "Improve credit score, maintain employment stability, keep debt low",
                    "Increase the model ensemble score",
                    "Wait for the model version to update",
                    "There's no clear guidance on how to improve"
                ],
                "correct_with_good": "Improve credit score, maintain employment stability, keep debt low",
                "correct_with_none": "There's no clear guidance on how to improve",
                "correct_with_poor": "There's no clear guidance on how to improve"
            }
        ],
        "insight": "Good explanations identify the key human-understandable factors (credit score, employment, debt ratio) that users can act on. Poor explanations provide technical model outputs that don't help users understand or improve."
    },
    {
        "title": "Resume Screening Result",
        "decision": "Your application did not advance to the interview stage",
        "versions": {
            "no_explanation": {
                "text": "Thank you for your interest in the Senior Software Engineer position. After careful review, we have decided not to move forward with your application at this time. We encourage you to apply for future positions that match your qualifications."
            },
            "good_explanation": {
                "text": "Thank you for your interest in the Senior Software Engineer position. After careful review, we have decided not to move forward with your application at this time.\n\nKey factors in our decision:\n• The position requires 7+ years of Python experience; your resume shows 4 years\n• We're looking for candidates with machine learning deployment experience, which wasn't evident in your application\n• Your frontend development skills are strong but this role focuses on backend systems\n\nYour qualifications are impressive, particularly your React expertise and project management experience. We encourage you to apply for future positions that better match your background, especially our Frontend Engineering roles.\n\nYou can update your profile at careers.company.com to receive alerts for relevant positions."
            },
            "poor_explanation": {
                "text": "Thank you for your interest in the Senior Software Engineer position. After careful review, we have decided not to move forward with your application at this time.\n\nApplication Analysis:\n• Skills match score: 0.62 (threshold: 0.75)\n• Experience alignment vector: [0.57, 0.41, 0.68, 0.89, 0.23, ...]\n• Semantic resume embedding cosine similarity: 0.634\n• Keyword extraction match rate: 47/82 terms\n• Contextual BERT model confidence: 0.592\n• Relative ranking: 47th percentile of applicant pool\n• Model disagreement index: 0.18\n• NLP sentiment score: 0.76 (neutral-positive)\n\nWe encourage you to apply for future positions that match your qualifications."
            }
        },
        "comprehension_questions": [
            {
                "question": "What specific skills gap led to the rejection?",
                "options": [
                    "Insufficient Python experience and lack of ML deployment experience",
                    "Low skills match score and semantic similarity",
                    "General mismatch with the position",
                    "The explanation doesn't make this clear"
                ],
                "correct_with_good": "Insufficient Python experience and lack of ML deployment experience",
                "correct_with_none": "The explanation doesn't make this clear",
                "correct_with_poor": "The explanation doesn't make this clear"
            },
            {
                "question": "What roles would be a better fit for this applicant?",
                "options": [
                    "Frontend Engineering roles that utilize their React expertise",
                    "Positions with higher keyword match rates",
                    "Jobs requiring higher BERT confidence scores",
                    "The explanation doesn't provide this guidance"
                ],
                "correct_with_good": "Frontend Engineering roles that utilize their React expertise",
                "correct_with_none": "The explanation doesn't provide this guidance",
                "correct_with_poor": "The explanation doesn't provide this guidance"
            }
        ],
        "insight": "Good explanations translate rejection into actionable career guidance (what skills to develop, what roles to pursue). Poor explanations dump technical scores that don't help candidates understand what went wrong or how to improve."
    },
    {
        "title": "Content Moderation Decision",
        "decision": "Your post has been removed for violating community guidelines",
        "versions": {
            "no_explanation": {
                "text": "Your post has been removed for violating community guidelines. If you believe this was a mistake, you can appeal this decision."
            },
            "good_explanation": {
                "text": "Your post has been removed for violating community guidelines.\n\nSpecific issue:\nYour post contained language that our system identified as potentially harassing toward another user. Specifically, the phrase 'you people always...' followed by a generalization was flagged as targeting a group based on identity.\n\nWhy this matters:\nOur community standards prohibit generalizations that stereotype or demean groups of people, even in heated discussions.\n\nWhat you can do:\n• Review our community guidelines at community.com/guidelines\n• Edit your post to remove the problematic language and resubmit\n• If you believe this was removed in error, appeal at community.com/appeals\n\nWe value constructive dialogue and want to help you participate while keeping our community respectful."
            },
            "poor_explanation": {
                "text": "Your post has been removed for violating community guidelines.\n\nModeration System Analysis:\n• Toxicity classifier score: 0.782 (threshold: 0.65)\n• Hate speech probability: 0.341\n• Harassment probability: 0.689\n• Identity attack probability: 0.512\n• Profanity detection: 0.089\n• Threat classification: 0.034\n• Spam likelihood: 0.012\n• Context embedding similarity to policy violations: 0.723\n• User history risk factor: 0.15\n• Ensemble decision: REMOVE (confidence: 0.81)\n\nIf you believe this was a mistake, you can appeal this decision."
            }
        },
        "comprehension_questions": [
            {
                "question": "What specifically in the post violated the guidelines?",
                "options": [
                    "Language that stereotyped a group ('you people always...')",
                    "High toxicity and harassment scores",
                    "Multiple policy violation indicators",
                    "The explanation doesn't specify"
                ],
                "correct_with_good": "Language that stereotyped a group ('you people always...')",
                "correct_with_none": "The explanation doesn't specify",
                "correct_with_poor": "The explanation doesn't specify"
            },
            {
                "question": "How can the user participate without violating guidelines?",
                "options": [
                    "Remove stereotyping language and focus on specific behaviors, not groups",
                    "Reduce toxicity and harassment scores",
                    "Lower the ensemble decision confidence",
                    "The explanation doesn't provide clear guidance"
                ],
                "correct_with_good": "Remove stereotyping language and focus on specific behaviors, not groups",
                "correct_with_none": "The explanation doesn't provide clear guidance",
                "correct_with_poor": "The explanation doesn't provide clear guidance"
            }
        ],
        "insight": "Good moderation explanations specify the problematic content and teach users the underlying principle. Poor explanations show technical scores without helping users understand what they did wrong or how to communicate acceptably."
    }
]

In [ ]:
#@title Explanation Validator
class ExplanationValidator:
    def __init__(self, scenarios_list):
        self.scenarios = scenarios_list.copy()
        self.current_scenario = 0
        self.current_version = None
        self.version_order = []
        self.results = []
        self.question_index = 0
        self.current_answers = []
        
        random.shuffle(self.scenarios)
        
        # Create widgets
        self.scenario_display = HTML(value="<h3>Explanation Validator</h3>")
        self.explanation_display = HTML(value="")
        self.question_display = HTML(value="")
        
        self.answer_selector = RadioButtons(
            options=[],
            description='Answer:',
            disabled=False,
            layout=Layout(width='100%')
        )
        
        self.submit_answer_button = Button(
            description='Submit Answer',
            button_style='primary',
            layout=Layout(width='200px', height='40px', margin='10px')
        )
        
        self.next_version_button = Button(
            description='Try Next Version →',
            button_style='info',
            layout=Layout(width='200px', height='40px', margin='10px')
        )
        
        self.feedback_display = HTML(value="")
        self.progress_display = HTML(value="")
        
        # Events
        self.submit_answer_button.on_click(lambda b: self.submit_answer())
        self.next_version_button.on_click(lambda b: self.next_version())
        
        # Layout
        submit_box = HBox([self.submit_answer_button], layout=Layout(justify_content='center'))
        next_box = HBox([self.next_version_button], layout=Layout(justify_content='center'))
        
        self.widget = VBox([
            self.scenario_display,
            self.explanation_display,
            self.question_display,
            self.answer_selector,
            submit_box,
            self.feedback_display,
            next_box,
            self.progress_display
        ])
    
    def start_validator(self):
        self.current_scenario = 0
        self.results = []
        self.show_scenario()
        return self.widget
    
    def show_scenario(self):
        if self.current_scenario >= len(self.scenarios):
            self.show_final_results()
            return
        
        # Randomize order of versions
        self.version_order = ['no_explanation', 'good_explanation', 'poor_explanation']
        random.shuffle(self.version_order)
        
        self.current_version_index = 0
        self.current_answers = []
        
        scenario = self.scenarios[self.current_scenario]
        
        self.scenario_display.value = f"""
        <div style='padding: 20px; background: #f8f9fa; border-radius: 10px; margin: 15px 0;'>
            <h3 style='color: #2c3e50; margin-bottom: 15px;'>Scenario {self.current_scenario + 1}: {scenario['title']}</h3>
            <div style='background: #d1ecf1; padding: 12px; border-radius: 8px;'>
                <p style='color: #0c5460; margin: 0; font-weight: bold;'>📊 You'll see the same AI decision presented in 3 different ways. For each version, answer comprehension questions to test your understanding. We'll compare how well each explanation helps you understand.</p>
            </div>
        </div>
        """
        
        self.show_version()
    
    def show_version(self):
        scenario = self.scenarios[self.current_scenario]
        version_key = self.version_order[self.current_version_index]
        version_data = scenario['versions'][version_key]
        
        # Don't reveal which version type it is yet
        self.explanation_display.value = f"""
        <div style='border: 2px solid #007bff; padding: 20px; border-radius: 10px; margin: 15px 0; background: white;'>
            <h4 style='color: #007bff; margin: 0 0 15px 0;'>Version {self.current_version_index + 1} of 3</h4>
            <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; white-space: pre-wrap; line-height: 1.6;'>{version_data['text']}</div>
        </div>
        """
        
        self.question_index = 0
        self.show_question()
    
    def show_question(self):
        scenario = self.scenarios[self.current_scenario]
        
        if self.question_index >= len(scenario['comprehension_questions']):
            self.finish_version()
            return
        
        question = scenario['comprehension_questions'][self.question_index]
        
        self.question_display.value = f"""
        <div style='padding: 15px; background: #fff3cd; border-radius: 8px; margin: 10px 0;'>
            <h4 style='color: #856404; margin: 0 0 10px 0;'>Question {self.question_index + 1} of {len(scenario['comprehension_questions'])}</h4>
            <p style='margin: 0; font-size: 16px;'><strong>{question['question']}</strong></p>
        </div>
        """
        
        self.answer_selector.options = question['options']
        self.answer_selector.value = None
        self.answer_selector.disabled = False
        self.submit_answer_button.disabled = False
        self.submit_answer_button.layout.display = 'inline-block'
        self.next_version_button.layout.display = 'none'
        self.feedback_display.value = ""
        
        self.update_progress()
    
    def submit_answer(self):
        if not self.answer_selector.value:
            self.feedback_display.value = "<p style='color: #f39c12;'>Please select an answer</p>"
            return
        
        scenario = self.scenarios[self.current_scenario]
        question = scenario['comprehension_questions'][self.question_index]
        version_key = self.version_order[self.current_version_index]
        
        # Determine correct answer based on version type
        if version_key == 'good_explanation':
            correct_answer = question['correct_with_good']
        elif version_key == 'no_explanation':
            correct_answer = question['correct_with_none']
        else:  # poor_explanation
            correct_answer = question['correct_with_poor']
        
        is_correct = self.answer_selector.value == correct_answer
        
        self.current_answers.append({
            'question_index': self.question_index,
            'answer': self.answer_selector.value,
            'correct': is_correct
        })
        
        color = "#27ae60" if is_correct else "#e74c3c"
        result = "✓ Correct!" if is_correct else "✗ Incorrect"
        
        self.feedback_display.value = f"""
        <div style='border: 2px solid {color}; padding: 12px; border-radius: 8px; margin: 10px 0; background: {color}15;'>
            <p style='color: {color}; margin: 0; font-weight: bold;'>{result}</p>
            <p style='margin: 5px 0 0 0;'>Correct answer: {correct_answer}</p>
        </div>
        """
        
        self.answer_selector.disabled = True
        self.submit_answer_button.disabled = True
        
        self.question_index += 1
        
        # Auto-advance to next question after brief delay
        if self.question_index < len(scenario['comprehension_questions']):
            self.submit_answer_button.description = 'Next Question →'
            self.submit_answer_button.disabled = False
            self.submit_answer_button.on_click(lambda b: self.show_question(), remove=True)
        else:
            self.finish_version()
    
    def finish_version(self):
        version_key = self.version_order[self.current_version_index]
        correct_count = sum(1 for a in self.current_answers if a['correct'])
        total = len(self.current_answers)
        
        self.results.append({
            'scenario': self.scenarios[self.current_scenario]['title'],
            'version': version_key,
            'score': correct_count,
            'total': total
        })
        
        self.question_display.value = f"""
        <div style='padding: 15px; background: #d4edda; border-radius: 8px; margin: 10px 0;'>
            <h4 style='color: #155724; margin: 0;'>Version {self.current_version_index + 1} Complete</h4>
            <p style='margin: 5px 0 0 0;'>You answered {correct_count} out of {total} questions correctly.</p>
        </div>
        """
        
        self.submit_answer_button.layout.display = 'none'
        
        self.current_version_index += 1
        
        if self.current_version_index < 3:
            self.next_version_button.layout.display = 'inline-block'
        else:
            self.show_scenario_results()
    
    def next_version(self):
        self.show_version()
    
    def show_scenario_results(self):
        scenario = self.scenarios[self.current_scenario]
        
        # Get results for this scenario
        scenario_results = [r for r in self.results if r['scenario'] == scenario['title']]
        
        # Organize by version type
        results_by_type = {}
        for r in scenario_results:
            results_by_type[r['version']] = r['score'] / r['total'] * 100
        
        # Create comparison
        comparison_html = """
        <div style='border: 3px solid #6f42c1; padding: 20px; border-radius: 10px; margin: 15px 0; background: #f3e5f5;'>
            <h3 style='color: #6f42c1; margin: 0 0 15px 0;'>📊 Explanation Effectiveness Comparison</h3>
            <p><strong>Your comprehension scores with each version:</strong></p>
        """
        
        version_labels = {
            'no_explanation': 'No Explanation',
            'good_explanation': 'Well-Designed Explanation',
            'poor_explanation': 'Technical Jargon Explanation'
        }
        
        for version_key in ['no_explanation', 'good_explanation', 'poor_explanation']:
            score = results_by_type.get(version_key, 0)
            label = version_labels[version_key]
            
            bar_color = "#27ae60" if version_key == 'good_explanation' else "#95a5a6"
            
            comparison_html += f"""
            <div style='margin: 10px 0;'>
                <p style='margin: 0 0 5px 0;'><strong>{label}:</strong> {score:.0f}%</p>
                <div style='background: #e0e0e0; border-radius: 5px; height: 25px;'>
                    <div style='background: {bar_color}; width: {score}%; height: 100%; border-radius: 5px;'></div>
                </div>
            </div>
            """
        
        comparison_html += f"""
            <div style='margin-top: 20px; padding: 15px; background: #fff9e6; border-radius: 8px;'>
                <p style='margin: 0; color: #856404;'><strong>💡 Key Insight:</strong> {scenario['insight']}</p>
            </div>
        </div>
        """
        
        self.scenario_display.value += comparison_html
        self.explanation_display.value = ""
        self.question_display.value = ""
        self.answer_selector.layout.display = 'none'
        
        self.current_scenario += 1
        
        if self.current_scenario < len(self.scenarios):
            self.next_version_button.description = 'Next Scenario →'
            self.next_version_button.layout.display = 'inline-block'
            self.next_version_button.on_click(lambda b: self.show_scenario(), remove=True)
        else:
            self.next_version_button.layout.display = 'none'
            self.show_final_results()
    
    def show_final_results(self):
        # Calculate average scores by version type
        scores_by_type = {'no_explanation': [], 'good_explanation': [], 'poor_explanation': []}
        
        for r in self.results:
            scores_by_type[r['version']].append(r['score'] / r['total'] * 100)
        
        avg_scores = {}
        for version_type, scores in scores_by_type.items():
            avg_scores[version_type] = sum(scores) / len(scores) if scores else 0
        
        self.scenario_display.value = f"""
        <div style='border: 3px solid #4CAF50; padding: 30px; border-radius: 15px; background: #f8fff8; text-align: center;'>
            <h2 style='color: #4CAF50; margin: 0 0 20px 0;'>Explanation Validation Complete! 📊</h2>
            
            <h3>Average Comprehension Scores</h3>
            <div style='max-width: 600px; margin: 20px auto;'>
                <div style='margin: 15px 0;'>
                    <p style='margin: 0 0 5px 0; text-align: left;'><strong>No Explanation:</strong> {avg_scores['no_explanation']:.0f}%</p>
                    <div style='background: #e0e0e0; border-radius: 5px; height: 30px;'>
                        <div style='background: #95a5a6; width: {avg_scores['no_explanation']}%; height: 100%; border-radius: 5px;'></div>
                    </div>
                </div>
                <div style='margin: 15px 0;'>
                    <p style='margin: 0 0 5px 0; text-align: left;'><strong>Well-Designed Explanation:</strong> {avg_scores['good_explanation']:.0f}%</p>
                    <div style='background: #e0e0e0; border-radius: 5px; height: 30px;'>
                        <div style='background: #27ae60; width: {avg_scores['good_explanation']}%; height: 100%; border-radius: 5px;'></div>
                    </div>
                </div>
                <div style='margin: 15px 0;'>
                    <p style='margin: 0 0 5px 0; text-align: left;'><strong>Technical Jargon Explanation:</strong> {avg_scores['poor_explanation']:.0f}%</p>
                    <div style='background: #e0e0e0; border-radius: 5px; height: 30px;'>
                        <div style='background: #e67e22; width: {avg_scores['poor_explanation']}%; height: 100%; border-radius: 5px;'></div>
                    </div>
                </div>
            </div>
            
            <h3>Skill Developed: Empirical Validation of Explanations</h3>
            <ul style='text-align: left; max-width: 700px; margin: 20px auto;'>
                <li><strong>Testing Over Assuming:</strong> Learned to validate that explanations actually improve understanding</li>
                <li><strong>Quality Recognition:</strong> Experienced the difference between good and poor explanations firsthand</li>
                <li><strong>Technical Translation:</strong> Saw how jargon-heavy explanations fail to help users</li>
                <li><strong>Measurement Mindset:</strong> Practiced using comprehension tests to evaluate explanation effectiveness</li>
            </ul>
            
            <h4>Key Insight</h4>
            <p style='max-width: 700px; margin: 0 auto;'>Providing an explanation doesn't automatically mean users will understand better.
            Poor explanations (technical jargon, model scores, irrelevant details) can actually reduce comprehension compared to
            no explanation at all. Good explanations translate AI decisions into human-understandable factors, provide actionable
            guidance, and help users learn. The only way to know if an explanation works is to test it empirically with real users.</p>
        </div>
        """
        
        self.explanation_display.value = ""
        self.question_display.value = ""
        self.answer_selector.layout.display = 'none'
        self.submit_answer_button.layout.display = 'none'
        self.next_version_button.layout.display = 'none'
        self.feedback_display.value = ""
    
    def update_progress(self):
        total_scenarios = len(self.scenarios)
        completed_scenarios = self.current_scenario
        
        self.progress_display.value = f"""
        <div style='background: #e8f4fd; padding: 12px; border-radius: 8px; text-align: center; margin-top: 15px;'>
            <strong>Progress:</strong> Scenario {self.current_scenario + 1}/{total_scenarios} |
            Version {self.current_version_index + 1}/3 |
            Question {self.question_index + 1}/{len(self.scenarios[self.current_scenario]['comprehension_questions'])}
        </div>
        """

In [ ]:
#@title Start the Module
# Start the Explanation Validator
validator = ExplanationValidator(validation_scenarios)
widget = validator.start_validator()
display(widget)